# Ablation: Batch Size (BS)

Comparing different batch sizes (with linearly scaled learning rate):
- **BS=32 (LR=1.3e-6)**: Small batch
- **BS=64 (LR=2.5e-6)**: Medium batch
- **BS=128 (LR=5e-6) (Default)**: Default batch size (baseline)

Note: Learning rate is scaled linearly with batch size following the linear scaling rule.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import numpy as np
from pathlib import Path

# Import shared ablation utilities
from ablation_utils import (
    setup_plotting_style,
    load_all_ablation_models,
    load_all_models_all_metrics,
    make_latex_ablation_table,
    plot_ablation_line,
    plot_ablation_bars,
    compute_deltas,
    print_summary,
    METRICS, METRIC_DISPLAY, METRIC_COLORS
)

# Set up plotting style
setup_plotting_style()

In [ ]:
# =============================================================================
# CONFIGURATION - Define ablation models
# =============================================================================

ABLATION_MODELS = {
    "BS=32": {
        "csv_path": "../evaluation/ablations/01-Jan_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr1ee-6_wd1e-2_neg_rel0.2_inplace1.0_swap1.0_ablation_bs32_lr1.3e-06.csv",
        "is_baseline": False,
        "description": "Small batch (LR=1.3e-6)",
        "batch_size": 32,
        "lr": 1.3e-6
    },
    "BS=64": {
        "csv_path": "../evaluation/ablations/01-Jan_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr3ee-6_wd1e-2_neg_rel0.2_inplace1.0_swap1.0_ablation_bs64_lr2.5e-06.csv",
        "is_baseline": False,
        "description": "Medium batch (LR=2.5e-6)",
        "batch_size": 64,
        "lr": 2.5e-6
    },
    "CS-CLIP (BS=128)": {
        "csv_path": "../evaluation/exp_csv/19-Dec_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr5ee-6_wd1e-2_neg_rel0.0_inplace1.0_swap1.0_csclip-negclip-hard-new_cleaned.csv",
        "is_baseline": True,
        "description": "Default batch size (LR=5e-6)",
        "batch_size": 128,
        "lr": 5e-6
    },
}

# Primary metric for comparison
PRIMARY_METRIC = "text_contrastive_accuracy"

# Checkpoint selection (use best or specific step)
CHECKPOINT_STEP = None  # None = use best checkpoint, or specify step like 5000

# Ablation metadata
ABLATION_NAME = "BATCH SIZE ABLATION"
PARAM_KEY = "batch_size"
PARAM_LABEL = 'Batch Size'

print("Ablation: Batch Size (BS)")
print("="*50)
for name, cfg in ABLATION_MODELS.items():
    baseline_mark = " [BASELINE]" if cfg["is_baseline"] else ""
    print(f"  {name}{baseline_mark}: {cfg['description']}")

In [ ]:
# =============================================================================
# LOAD DATA - Single Metric (Primary)
# =============================================================================

scores_df = load_all_ablation_models(ABLATION_MODELS, PRIMARY_METRIC, CHECKPOINT_STEP)
print(f"\nLoaded {len(scores_df)} models, {len(scores_df.columns)} datasets")

In [ ]:
# =============================================================================
# DISPLAY RAW SCORES TABLE
# =============================================================================

# Convert to percentage and display
scores_pct = scores_df * 100

# Add average column
scores_pct['Average'] = scores_pct.mean(axis=1)

print("\n" + "="*60)
print(f"ABLATION: BATCH SIZE")
print(f"Metric: {PRIMARY_METRIC}")
print("="*60)
display(scores_pct.round(1).style.highlight_max(axis=0, color='lightgreen'))

In [ ]:
# =============================================================================
# LOAD ALL METRICS (Text, Image, Group Contrastive Accuracy)
# =============================================================================

# Load all models with all metrics
all_metrics_df = load_all_models_all_metrics(ABLATION_MODELS, METRICS, CHECKPOINT_STEP)

# Extract just the summary columns (I2T, T2I, Group)
summary_cols = [col for col in ['I2T', 'T2I', 'Group'] if col in all_metrics_df.columns]
summary_df = all_metrics_df[summary_cols].copy()

# Add overall average
summary_df['Average'] = summary_df.mean(axis=1)

print("\n" + "="*60)
print("ABLATION: BATCH SIZE - ALL METRICS")
print("="*60)
display((summary_df * 100).round(1).style.highlight_max(axis=0, color='lightgreen'))

In [ ]:
# =============================================================================
# LATEX TABLE GENERATION
# =============================================================================

# Generate LaTeX table
latex_table = make_latex_ablation_table(
    summary_df,
    ABLATION_MODELS,
    caption="Batch size ablation (with linear LR scaling). I2T = Image-to-Text, T2I = Text-to-Image, Group = both correct. Best in \\textbf{bold}, baseline \\underline{underlined}.",
    label="tab:ablation_batch_size",
)

print("="*60)
print("LATEX TABLE")
print("="*60)
print(latex_table)

In [ ]:
# =============================================================================
# VISUALIZATION: LINE PLOT (BS vs Performance)
# =============================================================================

# Extract batch size values and corresponding scores
bs_values = [ABLATION_MODELS[model]['batch_size'] for model in summary_df.index]

fig, ax = plot_ablation_line(
    summary_df,
    bs_values,
    ABLATION_MODELS,
    param_label=PARAM_LABEL,
    title='Batch Size Ablation',
    save_path='../paper_figures/ablation_batch_size_line.pdf'
)

In [ ]:
# =============================================================================
# VISUALIZATION: GROUPED BAR CHART (All Metrics)
# =============================================================================

fig, ax = plot_ablation_bars(
    summary_df,
    ABLATION_MODELS,
    title='Batch Size Ablation',
    save_path='../paper_figures/ablation_batch_size_bars.pdf'
)

In [ ]:
# =============================================================================
# COMPUTE DELTAS FROM BASELINE
# =============================================================================

deltas_df = compute_deltas(summary_df, ABLATION_MODELS)

print("\n" + "="*60)
print("DELTA FROM BASELINE (percentage points)")
print("="*60)
display(deltas_df.round(2).style.background_gradient(cmap='RdYlGn', axis=None))

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print_summary(summary_df, ABLATION_MODELS, ABLATION_NAME, PARAM_KEY)

In [ ]:
# =============================================================================
# DATASET-WISE AND SUBSET-WISE TABLES (with ARO merging)
# =============================================================================

from ablation_utils import (
    load_all_models_per_dataset,
    load_all_models_per_subset,
    make_latex_dataset_table,
    get_datasets_and_subsets,
    display_all_tables,
    load_benchmark_config
)

# Load benchmark config for dataset merge rules (e.g., ARO)
bench_cfg = load_benchmark_config()

# Display all tables for the primary metric (I2T) with ARO merging
dataset_df, subset_df, datasets_subsets = display_all_tables(
    ABLATION_MODELS, PRIMARY_METRIC, CHECKPOINT_STEP, 
    show_latex=True, apply_merge=True, benchmark_config=bench_cfg
)